# 05｜上游动作包：跳跃与螃蟹舞

目标：查看现成控制器如何切换动作，理解训练策略与上游预训练动作包的区别。以下视频在 Ubuntu 工作站以 native CPU 和 OSMesa 实际录制。素材使用上游提交 `b2e58669848a3ba4f209597cd6bebdfe9d0533a4` 自带 `out/bundle_example/jumper.app`，该包 SHA-256 为 `454762cb9018a3ef91e29ef06dd350a44e7c934941a25cf289dec1cd2bcebc0b`。它不是本专题自训模型。

## 行走演示：前进指令与实际位移

这是上游预训练动作包的独立回放。在第 2–9 秒按住 W，指令逐渐增至 0.8 m/s；固定镜头便于观察位移，跟随镜头便于观察腿部。两段是独立回放，不能把两条轨迹合并。

In [1]:
from IPython.display import Video, display
display(Video('../assets/official-walk.mp4', embed=True, width=480))
display(Video('../assets/official-walk-follow.mp4', embed=True, width=480))

### 用采样轨迹计算位移

脚本每 0.05 秒采样一次机身世界坐标；取最接近 2 秒和 9 秒的点，计算水平直线位移及时间平均位移速度。该速度不是逐时速度均值。

In [2]:
import csv, math
from pathlib import Path
for filename in ('official-walk.csv', 'official-walk-follow.csv'):
    with (Path('../01-学习资料') / filename).open(encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    a, b = [min(rows, key=lambda r: abs(float(r['time_s']) - t)) for t in (2, 9)]
    dist = math.hypot(float(b['world_x_m']) - float(a['world_x_m']), float(b['world_y_m']) - float(a['world_y_m']))
    duration = float(b['time_s']) - float(a['time_s'])
    print(f'{filename}: {duration:.2f} s, {dist:.3f} m, {dist/duration:.3f} m/s')

official-walk.csv: 7.00 s, 4.461 m, 0.637 m/s
official-walk-follow.csv: 7.00 s, 4.348 m, 0.621 m/s


[固定镜头脚本](../02-可运行代码/official_walk_demo.py) · [跟随镜头脚本](../02-可运行代码/official_walk_follow_demo.py) · [固定镜头数据](../01-学习资料/official-walk.csv) · [跟随镜头数据](../01-学习资料/official-walk-follow.csv)

脚本在上游源码根目录运行，使用内部键盘注入接口；输出写入 `/tmp`。此演示使用上游动作包，不是自训策略。

## 跳跃演示

仿真 2.00 秒注入 Space，控制器从 locomotion 切换到 jump，3.79 秒返回 locomotion。画面可见离地。

In [3]:
from IPython.display import Video, display
display(Video('../assets/official-jump.mp4', embed=True, width=480))

## 螃蟹舞演示

仿真 2.00 秒注入 Ctrl+1，控制器切换到 dance_crab；画面后段可见挥爪。

In [4]:
display(Video('../assets/official-dance.mp4', embed=True, width=480))

## 在工作站重放

脚本使用上游内部 `AppPlayer._person` 接口模拟按键，绑定上述提交，接口随上游版本可能变化。将 [跳跃脚本](../02-可运行代码/official_jump_demo.py) 上传到工作站 `/tmp/official_jump_demo.py`，再从 Jumper 源码目录运行：

```bash
cd /data/Data14TB/robotics_shared/repos/jumper-everyembodied
MUJOCO_GL=osmesa OMP_NUM_THREADS=1 /data/Data14TB/envs/jumper-everyembodied/bin/python /tmp/official_jump_demo.py
```

脚本输出 `/tmp/jumper-official-jump.mp4`；[舞蹈脚本](../02-可运行代码/official_dance_demo.py) 同法运行，输出 `/tmp/jumper-official-dance.mp4`。两段视频各 9.03 秒。查看 [跳跃日志](../01-学习资料/official-jump.txt) 与 [舞蹈日志](../01-学习资料/official-dance.txt)，寻找模式切换时间。上游包内容与许可见 [Jumper](https://github.com/KingKongRobotics/jumper)。